$$\textit{To learn is virtue, to seek is divine}$$

In [3]:
# An equivalent way to implement this, is to set abs_
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import torch
import torch.nn.functional as F
from sorl.selfroute import _find_similar_magnitude_dims

nl_vocab  = int(model.vocab_sizes[0].item())
abs_vocab = int(model.vocab_sizes[1].item())   # includes placeholder at offset 0
n_abs_real = abs_vocab - 1                     # real abstract tokens (skip placeholder)

# (1). Initialize abstract lm_head rows: "similar magnitude" one-hot projection
with torch.no_grad():
    lm_w = model.model.lm_head.weight                      # (total_vocab, hidden)
    selected_dims, importances, cv = _find_similar_magnitude_dims(
        lm_w[:nl_vocab].float(), n_abs_real)
    print(f"Selected {n_abs_real} dims | "
          f"importance=[{importances.min():.4f}, {importances.max():.4f}] | CV={cv:.6f}")

    lm_w.data[nl_vocab:] = 0.0                             # zero all abstract rows
    for k in range(1, abs_vocab):                          # skip placeholder (k=0)
        lm_w.data[nl_vocab + k, selected_dims[k - 1].item()] = 1.0

print(f"lm_head rows [{nl_vocab}:{nl_vocab + abs_vocab}] initialized (similar-magnitude one-hot)")

Selected 128 dims | importance=[10.8605, 10.9294] | CV=0.001870
lm_head rows [151936:152065] initialized (similar-magnitude one-hot)


In [ ]:
# REINFORCE-style SoRL
# Policy  π(a | ctx): abstract token selection during recursion
# Reward  R_i = -traj_loss_i  (higher = NL tokens predicted better given abstractions)
# Baseline: group mean per sample  (REINFORCE with leave-one-out)
# Loss    = -E[ A_i · log π(a_i | ctx) ]   where A_i = (R_i - mean) / std

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sorl.sorl_trainer import (
    infer_insert_mask, expand_prompt_len, insert_tokens_with_padding,
)
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# ── Config ──────────────────────────────────────────────────────────────────
N  = 4          # rollouts per sample
K  = 4          # abstract tokens inserted
MAX_ITERS   = 2
TEMPERATURE = 1.0
MEM_SPAN    = 1792
LR          = 1e-5
BATCH_SIZE  = 2
LOG_EVERY   = 10
MAX_STEPS   = 300

dataset_name = "gsm8k"
max_length   = 256
# ────────────────────────────────────────────────────────────────────────────

train_ds = get_dataset(dataset_name, split="train", tokenizer=tokenizer, max_length=max_length)
val_ds   = get_dataset(dataset_name, split="test",  tokenizer=tokenizer, max_length=max_length)
dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

pad_id     = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history   = {"step": [], "loss": [], "reward_mean": [], "reward_std": [], "n_abs": []}

model.train()
global_step = 0
print(f"REINFORCE SoRL | K={K} | N={N} | temp={TEMPERATURE} | max_steps={MAX_STEPS}")
print("=" * 60)

for batch in dl:
    ids  = batch["input_ids"].to(device)
    attn = batch["attention_mask"].to(device)
    pl   = batch["prompt_len"].to(device)
    B    = ids.shape[0]

    # ── 1. Insert abstract token slots & run N rollouts (no grad) ───────────
    ins_mask = infer_insert_mask(ids, K, attn)
    exp_pl   = expand_prompt_len(pl, ins_mask)
    exp_data, exp_mask = insert_tokens_with_padding(
        ids, attn, ins_mask, model.vocab_sizes[0], pad_id)

    rep_data = exp_data.repeat_interleave(N, dim=0)    # (B*N, L)
    rep_mask = exp_mask.repeat_interleave(N, dim=0)
    rep_pl   = exp_pl.repeat_interleave(N, dim=0)

    with torch.no_grad():
        all_data, ppt, _ = model.recursion(
            rep_data, rep_mask,
            max_iterations=MAX_ITERS,
            memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN,
            temperature=TEMPERATURE, prompt_len=rep_pl,
        )
        # Reward = -mean NL traj loss per rollout  (shape: B*N)
        valid  = (ppt != 0).float()
        reward = -(ppt.sum(1) / valid.sum(1).clamp(min=1))

    break

    # ── 2. Group-normalised advantage (leave-one-out baseline) ──────────────
    r_g   = reward.view(B, N)
    mean_r = r_g.mean(1, keepdim=True)
    std_r  = r_g.std(1, keepdim=True).clamp(min=1e-6)
    adv    = ((r_g - mean_r) / std_r).view(-1)         # (B*N,)

    # ── 3. log π(abstract_choices | context)  under current policy ──────────
    outputs = model(
        input_ids=all_data, attention_mask=rep_mask,
        memory_span_abs=MEM_SPAN, memory_span_traj=MEM_SPAN,
    )
    shift_logits = outputs.logits[:, :-1, :].contiguous()   # (B*N, L-1, V)
    shift_ids    = all_data[:, 1:].contiguous()              # (B*N, L-1)
    shift_attn   = rep_mask[:, 1:].float()

    # Positions where an abstract token was placed
    abs_pos = (shift_ids >= base_vocab).float() * shift_attn  # (B*N, L-1)

    # Restrict softmax to abstract vocab; gather log prob of the chosen token
    abs_logits = shift_logits.clone()
    abs_logits[..., :base_vocab] = -float("inf")
    log_probs  = F.log_softmax(abs_logits, dim=-1)

    safe_ids   = shift_ids.clone()
    safe_ids[shift_ids < base_vocab] = base_vocab              # safe gather index
    per_tok_lp = log_probs.gather(2, safe_ids.unsqueeze(-1)).squeeze(-1)  # (B*N, L-1)
    per_tok_lp = per_tok_lp * abs_pos

    n_abs_toks = abs_pos.sum(1).clamp(min=1)
    log_pi     = per_tok_lp.sum(1) / n_abs_toks               # (B*N,)

    # ── 4. REINFORCE update ─────────────────────────────────────────────────
    loss = -(adv.detach() * log_pi).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    global_step += 1

    if global_step % LOG_EVERY == 0:
        r_mean = reward.mean().item()
        r_std  = reward.std().item()
        n_abs_avg = abs_pos.sum(1).mean().item()
        print(f"step {global_step:4d} | loss={loss.item():.4f} "
              f"| reward μ={r_mean:.3f} σ={r_std:.3f} | abs_toks/seq={n_abs_avg:.1f}")
        history["step"].append(global_step)
        history["loss"].append(loss.item())
        history["reward_mean"].append(r_mean)
        history["reward_std"].append(r_std)
        history["n_abs"].append(n_abs_avg)

    if global_step >= MAX_STEPS:
        break

print(f"\nDone. {global_step} steps.")